# Batch nucleus segmentation for INDI production

## Experimental metadata extraction

### Metadata from epxerimental design

In [3]:
# load in metadata files
import pandas as pd

automated_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_automated_plates_combined.csv")
manual_plates = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/all_manual_plates_combined.csv")
plate_id = pd.read_csv("/data/CARDPB2/iNDI/Production/metadata/indi_plateID_to_folderID.csv")

In [24]:
import pandas as pd

experimental_design = pd.read_csv("/data/kelpschdj/iNDI/Ab_combo_testing/Experimental_design.csv")
experimental_design.head()

,Row_alpha,Row,Column,AbComb_iNDI,DAPI,CF488A,CF568,CF640
0,B,2,7,1,DAPI,GM130-CF488A,LAMP1,G3BP1
1,C,3,7,1,DAPI,GM130-CF488A,LAMP1,G3BP1
2,D,4,7,1,DAPI,GM130-CF488A,LAMP1,G3BP1
3,E,5,7,1,DAPI,GM130-CF488A,LAMP1,G3BP1
4,F,6,7,1,DAPI,GM130-CF488A,LAMP1,G3BP1


In [25]:
# consider adding a new column that representes the organelle targeted by the antibody

experimental_design = experimental_design.melt(
    id_vars=["Row_alpha", "Row", "Column", "AbComb_iNDI"], 
    value_vars=["DAPI", "CF488A", "CF568", "CF640"],
    var_name="Channel_name",
    value_name="Stain")

In [26]:
print(experimental_design)

    Row_alpha  Row  Column  AbComb_iNDI Channel_name  Stain
0           B    2       7            1         DAPI   DAPI
1           C    3       7            1         DAPI   DAPI
2           D    4       7            1         DAPI   DAPI
3           E    5       7            1         DAPI   DAPI
4           F    6       7            1         DAPI   DAPI
..        ...  ...     ...          ...          ...    ...
611         K   11      17           11        CF640   MAP2
612         L   12      17           11        CF640   MAP2
613         M   13      17           11        CF640   MAP2
614         N   14      17           12        CF640  TUBB3
615         O   15      17           12        CF640  TUBB3

[616 rows x 6 columns]


## Opera Phenix metadata
Marianita collected images using the Opera Phenix. She ran the experiment twice, once where every channel was captured at the same exact z-plane and another where the nuclear channel was captured at a different z-height to more [potentially] represent the nuclei better. 

| Opera Phenix ID                      | Plate ID        | Measurement ID          | Measurement start time            | Notes                                          |
| ------------------------------------ | --------------- | ----------------------- | --------------------------------- | ---------------------------------------------- |
| cd085047-3dc5-4f83-abd0-104120a0f98f | 250823_175350-V | 2ea6a241-62ac-4bae-a6d1 | 2025-08-23T12:59:59.1112971-04:00 | DAPI z is different height than other channels |
| 9dee1dc4-f36e-49a7-b015-d5445e05283a | 250822_231213-V | 918078c0-ed0a-46ea-bbcf | 2025-08-22T18:17:41.4493257-04:00 | All z's are the same height                    |

In [27]:
import pandas as pd

samplesheet = pd.read_csv("/data/kelpschdj/iNDI/Ab_combo_testing/cd085047-3dc5-4f83-abd0-104120a0f98f_samplesheet.csv")
print(samplesheet)

                                                filepath  \
0      /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
1      /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
2      /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
3      /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
4      /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
...                                                  ...   
60975  /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
60976  /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
60977  /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
60978  /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   
60979  /data/INDIimage/iNDI_Data/AbTesting/cd085047-3...   

                        filename subdirectory  Row  Column  Frame  Plane  \
0      r02c07f01p01-ch01t01.tiff       r02c07    2       7      1      1   
1      r02c07f01p01-ch02t01.tiff       r02c07    2       7      1      1   
2      r02c07f01p01-ch03t01.tiff       r02c07    2 

In [28]:
import dask
from dask import delayed, compute
import pandas as pd
import tifffile
import numpy as np
from skimage import filters, morphology
from scipy import ndimage as ndi
from skimage.measure import regionprops_table
from scipy.stats import norm
import os

# --- Parameters ---
intensity_scaling_param = [1, 7]
blur_sigma = 1
min_area = 3500

# --- Function to process a single image ---
@delayed
def process_nucleus_image(image_path):
    # Load image
    nuc = tifffile.imread(image_path)
    
    # Normalize
    m, s = norm.fit(nuc.flatten())
    stretch_min = max(m - intensity_scaling_param[0] * s, nuc.min())
    stretch_max = min(m + intensity_scaling_param[1] * s, nuc.max())
    nuc_stretch = np.clip(nuc, stretch_min, stretch_max)
    image_norm = (nuc_stretch - stretch_min) / (stretch_max - stretch_min)

    # Blur
    blurred = filters.gaussian(image_norm, sigma=blur_sigma)

    # Step 1: Low level threshold
    triangle_cutoff = filters.threshold_triangle(blurred)
    global_median_cutoff = np.percentile(blurred, 50)
    th_low_cutoff = (triangle_cutoff + global_median_cutoff) / 2
    img_low_level = blurred > th_low_cutoff
    img_low_level = morphology.remove_small_objects(img_low_level.astype(bool), min_size=min_area)
    img_low_level = morphology.dilation(img_low_level, footprint=morphology.disk(2))

    # Step 2: High level threshold
    otsu_cutoff = 0.333 * filters.threshold_otsu(blurred)
    img_high_level = np.zeros_like(img_low_level)
    lab_low, num_obj = morphology.label(img_low_level, return_num=True)
    for idx in range(num_obj):
        single_obj = lab_low == (idx + 1)
        local_otsu = filters.threshold_otsu(blurred[single_obj])
        if local_otsu > otsu_cutoff:
            img_high_level[np.logical_and(blurred > 0.98 * local_otsu, single_obj)] = 1

    filled = ndi.binary_fill_holes(img_high_level)
    filled = morphology.dilation(filled, footprint=morphology.disk(2))
    labeled_filled = morphology.label(filled)
    nuc_seg = morphology.remove_small_objects(labeled_filled, min_size=min_area)

    # Extract features
    props = regionprops_table(
        label_image=nuc_seg,
        intensity_image=nuc,
        properties=(
            'label', 'area', 'mean_intensity', 'max_intensity', 'min_intensity', 'std_intensity',
            'centroid', 'eccentricity', 'solidity', 'perimeter', 'feret_diameter_max',
            'orientation', 'major_axis_length', 'minor_axis_length'
        )
    )
    df = pd.DataFrame(props)
    df['image_name'] = os.path.basename(image_path)
    
    return df

In [29]:
# Filter only DAPI images
dapi_samples = samplesheet[samplesheet['Channel_name'] == "DAPI"].copy()

# Extract file paths
filepaths = dapi_samples['filepath'].tolist()

# --- Wrap all images with delayed ---
tasks = [process_nucleus_image(fp) for fp in filepaths]

In [ ]:
filepaths

In [30]:
from dask.diagnostics import ProgressBar
with ProgressBar():
    dfs = dask.compute(*tasks, scheduler="processes")

final_df = pd.concat(dfs, ignore_index=True)
final_df.to_csv("nuclei_features_20250930_cd.csv", index=False)


[                                        ] | 0% Completed | 12.73 sms

/tmp/ipykernel_607560/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[#####                                   ] | 14% Completed | 85.86 s

/tmp/ipykernel_607560/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[######                                  ] | 15% Completed | 93.27 s

...

[##################################      ] | 87% Completed | 473.56 s

/tmp/ipykernel_607560/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?
/tmp/ipykernel_607560/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[######################################  ] | 97% Completed | 530.01 s

/tmp/ipykernel_607560/988843639.py:54: UserWarning: Only one label was provided to `remove_small_objects`. Did you mean to use a boolean array?


[########################################] | 100% Completed | 541.66 s
